# Chapter 15: The Likelihood Ratio Test and Wilks' Theorem

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch15/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_15.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Verifying Wilks' Theorem (Poisson)

*Source file:* `code/ch15/c15_1_verifying_wilks_theorem_poisson.py`

In [2]:
import numpy as np
import scipy.stats as stats

rng = np.random.default_rng(42)
samples = rng.poisson(lam=4.0, size=(150_000, 100))
xb = np.mean(samples, axis=1)
g2 = 2.0 * 100 * (xb * np.log(xb / 4.0) - (xb - 4.0))
print(f"E[G^2]: {np.mean(g2):.4f} (theory: 1.0) | type I error: {np.mean(g2 > stats.chi2.ppf(0.95, 1)):.4f}")
print(f"Var[G^2]: {np.var(g2):.4f} (theory: 2.0) | KS p-value (against chi2_1): {stats.kstest(g2, 'chi2', args=(1,)).pvalue:.3f}")

E[G^2]: 0.9998 (theory: 1.0) | type I error: 0.0517
Var[G^2]: 2.0141 (theory: 2.0) | KS p-value (against chi2_1): 0.000


## The Holy Trinity: Exact Sizes in Small Samples

Since the Poisson sum $T=n\bar X\sim\text{Poisson}(n\lambda_0)$, the true sizes of the three tests under $H_0$ can be computed exactly, without simulation.

*Source file:* `code/ch15/c15_2_holy_trinity_exact_sizes.py`

In [3]:
import numpy as np
from scipy import stats

def sizes(n, lam0, alpha=0.05):
    t = np.arange(0, int(n*lam0 + 20*np.sqrt(n*lam0) + 30))
    w = stats.poisson.pmf(t, n*lam0)
    xb = t / n
    with np.errstate(divide="ignore", invalid="ignore"):
        G = 2*n*(np.where(xb > 0, xb*np.log(xb/lam0), 0.0) - (xb - lam0))
        W = np.where(xb > 0, n*(xb - lam0)**2 / xb, np.inf)
    S = n*(xb - lam0)**2 / lam0
    k = stats.chi2.ppf(1 - alpha, 1)
    return [np.sum(w[s > k]) for s in (G, W, S)]

print(" n   lambda0 |  LRT    Wald   Score  (target 0.05)")
for n, l0 in [(5, 1.0), (10, 1.0), (10, 0.3), (50, 1.0), (200, 1.0)]:
    g, w, s = sizes(n, l0)
    print(f"{n:4d}  {l0:5.1f}   | {g:.4f} {w:.4f} {s:.4f}")

 n   lambda0 |  LRT    Wald   Score  (target 0.05)
   5    1.0   | 0.0723 0.1301 0.0386
  10    1.0   | 0.0563 0.0743 0.0374
  10    0.3   | 0.0833 0.2030 0.0335
  50    1.0   | 0.0474 0.0512 0.0556


 200    1.0   | 0.0476 0.0537 0.0517


## Multinomial Goodness of Fit

*Source file:* `code/ch15/c15_3_multinomial_goodness_of_fit.py`

In [4]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
p0 = np.array([0.1, 0.2, 0.3, 0.4])
n, R = 40, 200_000
N = rng.multinomial(n, p0, size=R)
E = n * p0
with np.errstate(divide="ignore", invalid="ignore"):
    G2 = 2 * np.sum(np.where(N > 0, N * np.log(N / E), 0.0), axis=1)
X2 = np.sum((N - E)**2 / E, axis=1)
k = stats.chi2.ppf(0.95, df=3)
print(f"n={n}: P(G^2 > chi2_3,0.05) = {np.mean(G2 > k):.4f} | P(X^2 > chi2_3,0.05) = {np.mean(X2 > k):.4f}")
print(f"means: G^2={G2.mean():.3f}, X^2={X2.mean():.3f} (theory 3) | Corr(G^2, X^2)={np.corrcoef(G2, X2)[0,1]:.4f}")
# An example data set
observed = np.array([7, 5, 13, 15])
r1 = stats.chisquare(observed, E)
r2 = stats.power_divergence(observed, E, lambda_="log-likelihood")
print(f"Observed {observed}, expected {E}")
print(f"Pearson X^2 = {r1.statistic:.4f}, p = {r1.pvalue:.4f} | G^2 = {r2.statistic:.4f}, p = {r2.pvalue:.4f}")

n=40: P(G^2 > chi2_3,0.05) = 0.0600 | P(X^2 > chi2_3,0.05) = 0.0474
means: G^2=3.104, X^2=2.996 (theory 3) | Corr(G^2, X^2)=0.9666
Observed [ 7  5 13 15], expected [ 4.  8. 12. 16.]
Pearson X^2 = 3.5208, p = 0.3181 | G^2 = 3.2795, p = 0.3505


## Real Data: Deaths by Horse Kick in the Prussian Army

The data that Bortkiewicz compiled to illustrate his “law of small numbers”  are the most famous application of the Poisson distribution: the number of soldiers killed by horse kicks each year in 14 Prussian army corps, 1875–1894. Fisher left out four corps that were organised differently and used the 200 corps-years of the remaining 10. The code asks two questions. (i) Do the counts follow a Poisson distribution? Since $\lambda$ is estimated by the MLE, the degrees of freedom of the Pearson $X^2$ and $G^2$ statistics are the number of cells minus one, minus the number of estimated parameters: $4-1-1=2$. (Because $\lambda$ is estimated from the ungrouped data, the null distribution is not exactly $\chi^2_2$ but lies between $\chi^2_2$ and $\chi^2_3$ ; here the difference is negligible.) (ii) Do all corps produce deaths at the same rate? The 20-year total of corps $c$ is $T_c\sim\text{Poisson}(20\lambda_c)$. The LRT of the model with a separate rate for each corps against the common-rate model,
$$
G^2=2\sum_cT_c\ln\frac{T_c}{20\hat\lambda},
$$
is approximately $\chi^2_{K-1}$ by Wilks' theorem (Theorem 15.5), where $K$ is the number of corps.

*Source file:* `code/ch15/c15_4_horse_kicks_real_data.py`

In [5]:
import numpy as np
from scipy import stats

# Bortkiewicz (1898): soldiers killed by horse kicks in 14 Prussian army corps, 1875--1894.
# Source: R package vcd, data set VonBort. Rows are corps, columns are years.
corps = [
    'G', 'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X', 'XI', 'XIV', 'XV',
]
deaths = np.array([
    [0, 2, 2, 1, 0, 0, 1, 1, 0, 3, 0, 2, 1, 0, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 2, 0, 3, 0, 2, 0, 0, 0, 1, 1, 1, 0, 2, 0, 3, 1, 0],
    [0, 0, 0, 2, 0, 2, 0, 0, 1, 1, 0, 0, 2, 1, 1, 0, 0, 2, 0, 0],
    [0, 0, 0, 1, 1, 1, 2, 0, 2, 0, 0, 0, 1, 0, 1, 2, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 2, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 2, 0, 0, 1, 2, 0, 1, 1, 3, 1, 1, 1, 0, 3, 0, 0],
    [1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 2, 0, 0, 2, 1, 0, 2, 0],
    [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 2, 1, 1, 1, 0, 2, 1, 1, 0, 1, 2, 0, 1, 0, 0],
    [0, 0, 1, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0, 0, 2, 1, 3, 0, 1, 1],
    [0, 0, 0, 0, 2, 4, 0, 1, 3, 0, 1, 1, 1, 1, 2, 1, 3, 1, 3, 1],
    [1, 1, 2, 1, 1, 3, 0, 4, 0, 1, 0, 3, 2, 1, 0, 2, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 2, 2, 0, 0, 0, 0],
])
# the 10 corps used by Fisher (G, I, VI and XI, which were organised differently, are left out)
fisher = np.array([
    False, False, True, True, True, True, False, True, True, True, True, False, True, True,
])

def fit(x):
    lam = x.mean()                                         # Poisson MLE
    observed = np.array([np.sum(x == 0), np.sum(x == 1), np.sum(x == 2), np.sum(x >= 3)])
    p = stats.poisson.pmf([0, 1, 2], lam)
    expected = x.size * np.append(p, 1 - p.sum())
    X2 = np.sum((observed - expected)**2 / expected)
    G2 = 2 * np.sum(observed * np.log(observed / expected))
    return lam, observed, expected, X2, G2

for name, M in [("Fisher's 10 corps", deaths[fisher]), ("All 14 corps", deaths)]:
    lam, g, b, X2, G2 = fit(M.ravel())
    print(f"{name}: {M.size} corps-years, {M.sum()} deaths, lambda^ = {lam:.3f}")
    print(f"  observed (0, 1, 2, 3+) = {g} | expected = {np.round(b, 2)}")
    print(f"  fit: X^2 = {X2:.3f}, G^2 = {G2:.3f}, df = 2, p = {stats.chi2.sf(G2, 2):.3f}")
    totals = M.sum(axis=1)                                 # 20-year total per corps
    e = M.shape[1] * lam
    Gh = 2 * np.sum(totals * np.log(totals / e))           # common rate, or a separate rate per corps?
    print(f"  corps totals = {totals}")
    print(f"  homogeneity LRT: G^2 = {Gh:.2f}, df = {len(totals) - 1}, p = {stats.chi2.sf(Gh, len(totals) - 1):.4f}")

Fisher's 10 corps: 200 corps-years, 122 deaths, lambda^ = 0.610
  observed (0, 1, 2, 3+) = [109  65  22   4] | expected = [108.67  66.29  20.22   4.82]
  fit: X^2 = 0.324, G^2 = 0.328, df = 2, p = 0.849
  corps totals = [12 12  8 11 12  7 13 15 24  8]
  homogeneity LRT: G^2 = 15.58, df = 9, p = 0.0762
All 14 corps: 280 corps-years, 196 deaths, lambda^ = 0.700
  observed (0, 1, 2, 3+) = [144  91  32  13] | expected = [139.04  97.33  34.07   9.56]
  fit: X^2 = 1.952, G^2 = 1.835, df = 2, p = 0.399
  corps totals = [16 16 12 12  8 11 17 12  7 13 15 25 24  8]
  homogeneity LRT: G^2 = 26.14, df = 13, p = 0.0163
